# Reasoning Probe — Results Explorer

Load and visualize the output from `05_reasoning_probe.py`.
This notebook answers: where did the model match your reasoning, and where did it fall short?

In [ ]:
import json
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from pathlib import Path

# Load latest probe results
results_path = Path('../probe_results/results_latest.json')

with open(results_path) as f:
    data = json.load(f)

print(f"Checkpoint: {data['checkpoint']}")
print(f"Prompts evaluated: {data['num_prompts']}")
print(f"\nSummary scores:")
for k, v in data['summary_scores'].items():
    print(f"  {k}: {v}")

In [ ]:
# Build a flat DataFrame for analysis
rows = []
for r in data['results']:
    rows.append({
        'prompt_index': r['prompt_index'],
        'prompt_short': r['prompt'][:60] + '...',
        'structural': r['scores']['structural_similarity'],
        'depth': r['scores']['reasoning_depth'],
        'opinion': r['scores']['opinion_consistency'],
        'composite': r['scores']['composite'],
        'verdict': r['verdict'],
    })

df = pd.DataFrame(rows)
df

In [ ]:
# ── Plot 1: Score breakdown per prompt ──────────────────────────────────────
fig, ax = plt.subplots(figsize=(12, 5))

x = range(len(df))
width = 0.25

bars1 = ax.bar([i - width for i in x], df['structural'], width, label='Structural', color='#4a8fc4', alpha=0.85)
bars2 = ax.bar(x,                       df['depth'],      width, label='Depth',      color='#d4756b', alpha=0.85)
bars3 = ax.bar([i + width for i in x],  df['opinion'],    width, label='Opinion',    color='#6ab04c', alpha=0.85)

ax.set_xlabel('Prompt', fontsize=12)
ax.set_ylabel('Score (0-100)', fontsize=12)
ax.set_title('Reasoning Probe — Score Breakdown by Prompt', fontsize=14, fontweight='bold')
ax.set_xticks(list(x))
ax.set_xticklabels([f'P{i+1}' for i in x])
ax.set_ylim(0, 110)
ax.axhline(y=70, color='gray', linestyle='--', linewidth=0.8, label='Transfer threshold (70)')
ax.axhline(y=45, color='gray', linestyle=':', linewidth=0.8, label='Surface-only threshold (45)')
ax.legend(loc='upper right')
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
ax.set_facecolor('#f8f9fa')
fig.tight_layout()
plt.savefig('../probe_results/score_breakdown.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: probe_results/score_breakdown.png')

In [ ]:
# ── Plot 2: Radar chart — avg scores across three axes ──────────────────────
import numpy as np

categories = ['Structural\nSimilarity', 'Reasoning\nDepth', 'Opinion\nConsistency']
values = [
    data['summary_scores']['structural_similarity_avg'],
    data['summary_scores']['reasoning_depth_avg'],
    data['summary_scores']['opinion_consistency_avg'],
]

# Close the radar shape
values_plot = values + [values[0]]
angles = np.linspace(0, 2 * np.pi, len(categories), endpoint=False).tolist()
angles += angles[:1]

fig, ax = plt.subplots(figsize=(6, 6), subplot_kw=dict(polar=True))

ax.plot(angles, values_plot, 'o-', linewidth=2, color='#d4756b')
ax.fill(angles, values_plot, alpha=0.25, color='#d4756b')

# Add threshold ring
thresh = [70] * len(categories) + [70]
ax.plot(angles, thresh, '--', linewidth=1, color='gray', alpha=0.5, label='Transfer threshold')

ax.set_xticks(angles[:-1])
ax.set_xticklabels(categories, fontsize=11)
ax.set_ylim(0, 100)
ax.set_yticks([25, 50, 75, 100])
ax.set_yticklabels(['25', '50', '75', '100'], fontsize=8, color='gray')
ax.set_title('Average Reasoning Transfer\nacross all probe prompts', 
             fontsize=13, fontweight='bold', pad=20)
ax.legend(loc='upper right', bbox_to_anchor=(1.3, 1.1))

fig.tight_layout()
plt.savefig('../probe_results/radar_chart.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: probe_results/radar_chart.png')

In [ ]:
# ── Text analysis: where did the model fall shortest? ────────────────────────
print("PROMPTS WHERE REASONING DEPTH WAS LOWEST:\n")
worst = df.nsmallest(3, 'depth')[['prompt_short', 'structural', 'depth', 'opinion', 'composite']]
print(worst.to_string(index=False))

print("\nPROMPTS WHERE DEPTH WAS HIGHEST:\n")
best = df.nlargest(3, 'depth')[['prompt_short', 'structural', 'depth', 'opinion', 'composite']]
print(best.to_string(index=False))

print("\nOBSERVATION:")
print("High-scoring depth prompts likely match topics you've written about repeatedly.")
print("Low-scoring depth prompts are likely topics you've only touched once or twice.")
print("That's not a model problem. That's the corpus telling you something about your writing.")